# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 3: Evaluation, Baselines, Traditional ML

Today we'll write some simple models to predict the price of a product

We'll use an approach to evaluate the performance of the model

And we'll test some Baseline Models using Traditional machine learning

In [1]:
import random
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
from pricer.evaluator import evaluate
from pricer.items import Item

In [2]:
LITE_MODE = False

In [3]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


Generating test split: 100%|##########| 1000/1000 [00:00<00:00, 99966.73 examples/s]


In [4]:
def random_pricer(item):
    return random.randrange(1,1000)

In [5]:
random.seed(42)
evaluate(random_pricer, test)

$436 $1 $29 $690 $252 $21 $85 $72 $719 $225 $20 $380 $894 $505 $11 $572 $354 $17 $179 $23 $90 $115 $433 $442 $304 $122 $291 $714 $567 $639 $539 $370 $66 $380 $489 $534 $769 $835 $207 $740 $626 $84 $680 $178 $129 $260 $142 $189 $836 $580 $310 $25 $380 $270 $47 $234 $861 $313 $417 $259 $591 $33 $657 $361 $79 $38 $757 $500 $263 $5 $534 $284 $570 $625 $584 $871 $759 $361 $575 $178 $602 $60 $17 $579 $207 $732 $115 $224 $756 $193 $866 $9 $370 $250 $456 $423 $821 $217 $103 $195 $264 $98 $650 $135 $470 $842 $675 $264 $43 $325 $591 $138 $516 $619 $56 $2 $449 $369 $221 $845 $640 $616 $501 $59 $502 $273 $844 $688 $616 $81 $164 $705 $52 $795 $259 $396 $70 $2 $89 $798 $902 $331 $818 $716 $129 $186 $627 $2 $141 $873 $918 $553 $423 $7 $253 $136 $204 $707 $255 $502 $19 $739 $557 $426 $80 $575 $39 $321 $185 $132 $497 $484 $326 $751 $53 $733 $85 $64 $549 $71 $158 $672 $133 $362 $16 $373 $258 $544 $420 $515 $223 $944 $532 $743 $866 $68 $527 $459 $67 $673 

  0%|          | 0/200 [00:00<?, ?it/s]

In [6]:
# That was fun!
# We can do better - here's another rather trivial model

training_prices = [item.price for item in train]
training_average = sum(training_prices) / len(training_prices)
print(training_average)

def constant_pricer(item):
    return training_average

140.347293


In [7]:
evaluate(constant_pricer, test)

$79 $24 $85 $70 $110 $90 $4 $75 $105 $190 $573 $239 $120 $86 $61 $107 $61 $90 $70 $21 $6 $16 $55 $35 $192 $312 $355 $120 $42 $60 $120 $80 $20 $60 $25 $679 $81 $84 $73 $102 $59 $60 $105 $115 $80 $115 $122 $109 $5 $62 $105 $10 $335 $21 $87 $6 $133 $100 $62 $128 $96 $62 $49 $30 $488 $50 $100 $305 $15 $64 $109 $123 $140 $121 $90 $104 $16 $130 $123 $122 $20 $128 $110 $41 $113 $80 $42 $166 $20 $95 $118 $45 $120 $105 $132 $88 $106 $17 $130 $435 $40 $23 $103 $1 $109 $22 $115 $260 $109 $159 $80 $174 $109 $12 $55 $30 $116 $120 $84 $37 $124 $51 $70 $26 $60 $80 $120 $41 $39 $1 $69 $3 $55 $110 $75 $125 $65 $70 $12 $2 $76 $110 $60 $121 $54 $108 $95 $370 $125 $107 $121 $34 $93 $0 $121 $139 $91 $84 $179 $90 $149 $114 $98 $127 $700 $117 $307 $90 $100 $130 $115 $118 $280 $118 $39 $9 $112 $47 $46 $47 $514 $115 $160 $108 $90 $118 $7 $73 $80 $114 $105 $89 $105 $2 $40 $60 $30 $140 $89 $114 

  0%|          | 0/200 [00:00<?, ?it/s]

In [8]:
def get_features(item):
    return {
        "weight": item.weight,
        "weight_unknown": 1 if item.weight==0 else 0,
        "text_length": len(item.summary)
    }

In [9]:
def list_to_dataframe(items):
    features = [get_features(item) for item in items]
    df = pd.DataFrame(features)
    df['price'] = [item.price for item in items]
    return df

train_df = list_to_dataframe(train)
test_df = list_to_dataframe(test)

In [10]:
# Traditional Linear Regression!

np.random.seed(42)

# Separate features and target
feature_columns = ['weight', 'weight_unknown', 'text_length']

X_train = train_df[feature_columns]
y_train = train_df['price']
X_test = test_df[feature_columns]
y_test = test_df['price']

# Train a Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)

for feature, coef in zip(feature_columns, model.coef_):
    print(f"{feature}: {coef}")
print(f"Intercept: {model.intercept_}")

# Predict the test set and evaluate
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R-squared Score: {r2}")

weight: 3.5687444142553044
weight_unknown: 20.901759880237513
text_length: 0.2034312373915267
Intercept: 40.912387807911955
Mean Squared Error: 20096.925335647466
R-squared Score: 0.1609102308229894


In [11]:
def linear_regression_pricer(item):
    features = get_features(item)
    features_df = pd.DataFrame([features])
    return model.predict(features_df)[0]

In [12]:
evaluate(linear_regression_pricer, test)

$79 $51 $62 $66 $71 $50 $9 $50 $82 $191 $574 $233 $113 $58 $28 $97 $50 $62 $39 $14 $18 $14 $75 $11 $193 $318 $359 $91 $35 $36 $104 $78 $27 $16 $79 $676 $65 $66 $70 $67 $64 $36 $61 $66 $104 $93 $104 $83 $25 $60 $84 $5 $347 $2 $79 $20 $110 $83 $23 $123 $111 $49 $19 $3 $241 $17 $82 $261 $6 $68 $85 $105 $155 $89 $91 $80 $3 $98 $99 $88 $72 $86 $86 $8 $84 $43 $81 $176 $62 $60 $88 $42 $80 $71 $91 $117 $78 $8 $146 $418 $33 $7 $106 $21 $122 $1 $83 $298 $104 $198 $62 $154 $106 $59 $71 $53 $103 $94 $82 $14 $99 $43 $51 $51 $127 $47 $86 $102 $72 $16 $53 $9 $47 $72 $45 $84 $81 $40 $9 $11 $39 $132 $55 $85 $3 $71 $71 $364 $5 $72 $93 $26 $53 $21 $90 $119 $103 $40 $43 $79 $160 $92 $67 $116 $725 $100 $279 $74 $84 $101 $86 $100 $244 $78 $139 $58 $95 $7 $50 $38 $289 $117 $38 $94 $75 $96 $6 $77 $57 $79 $79 $78 $108 $21 $14 $97 $50 $109 $90 $114 

  0%|          | 0/200 [00:00<?, ?it/s]

In [13]:
prices = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [14]:
np.random.seed(42)
vectorizer = CountVectorizer(max_features=2000, stop_words='english')
X = vectorizer.fit_transform(documents)


In [15]:
# Here are the 1,000 most common words that it picked, not including "stop words":

selected_words = vectorizer.get_feature_names_out()
print(f"Number of selected words: {len(selected_words)}")
print("Selected words:", selected_words[1000:1020])

Number of selected words: 2000
Selected words: ['jack' 'jacket' 'jeep' 'jigsaw' 'joint' 'joints' 'keeping' 'keeps' 'key'
 'keyboard' 'keypad' 'keys' 'kg' 'khz' 'kia' 'kickstand' 'kids' 'king'
 'kingston' 'kit']


In [16]:
regressor = LinearRegression()
regressor.fit(X, prices)

Out[1]: LinearRegression()


In [17]:
def natural_language_linear_regression_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(regressor.predict(x)[0], 0)

In [18]:
evaluate(natural_language_linear_regression_pricer, test)

$86 $120 $55 $70 $189 $228 $25 $40 $8 $65 $480 $187 $60 $130 $21 $140 $64 $31 $25 $25 $39 $85 $50 $22 $318 $359 $149 $39 $36 $29 $75 $59 $4 $26 $118 $336 $10 $104 $164 $73 $150 $108 $40 $10 $90 $43 $73 $74 $15 $34 $41 $48 $153 $42 $79 $167 $112 $195 $13 $12 $39 $2 $52 $21 $427 $37 $27 $192 $8 $224 $20 $47 $91 $111 $16 $52 $140 $12 $22 $126 $8 $38 $72 $44 $59 $101 $5 $149 $53 $255 $29 $120 $84 $41 $52 $133 $65 $58 $212 $109 $89 $57 $37 $36 $3 $52 $11 $161 $31 $73 $19 $79 $206 $13 $35 $124 $119 $29 $49 $19 $16 $159 $8 $28 $62 $24 $20 $43 $59 $60 $8 $59 $178 $15 $26 $48 $1 $151 $101 $106 $49 $139 $4 $202 $139 $75 $18 $318 $18 $22 $19 $265 $28 $20 $100 $57 $163 $15 $102 $16 $64 $26 $42 $13 $386 $6 $84 $50 $1 $10 $9 $60 $319 $63 $71 $84 $10 $75 $1 $121 $386 $12 $64 $21 $108 $38 $6 $67 $151 $27 $41 $51 $29 $36 $32 $60 $45 $28 $12 $38 

  0%|          | 0/200 [00:00<?, ?it/s]

In [19]:
subset = 15_000
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
rf_model.fit(X[:subset], prices[:subset])

Out[1]: RandomForestRegressor(n_jobs=4, random_state=42)


## Random Forest model

The Random Forest is a type of "**ensemble**" algorithm, meaning that it combines many smaller algorithms to make better predictions.

It uses a very simple kind of machine learning algorithm called a **decision tree**. A decision tree makes predictions by examining the values of features in the input. Like a flow chart with IF statements. Decision trees are very quick and simple, but they tend to overfit.

In our case, the "features" are the elements of the Vector - in other words, it's the number of times that a particular word appears in the product description.

So you can think of it something like this:

**Decision Tree**  
\- IF the word "TV" appears more than 3 times THEN  
-- IF the word "LED" appears more than 2 times THEN  
--- IF the word "HD" appears at least once THEN  
---- Price = $500


With Random Forest, multiple decision trees are created. Each one is trained with a different random subset of the data, and a different random subset of the features. You can see above that we specify 100 trees, which is the default.

Then the Random Forest model simply takes the average of all its trees to product the final result.

In [20]:
def random_forest(item):
    x = vectorizer.transform([item.summary])
    return max(0, rf_model.predict(x)[0])

In [21]:
evaluate(random_forest, test)

$67 $62 $9 $14 $130 $145 $74 $58 $39 $234 $523 $291 $48 $82 $8 $2 $52 $93 $26 $62 $17 $25 $14 $32 $162 $307 $162 $122 $35 $49 $49 $85 $99 $18 $32 $365 $48 $20 $139 $87 $153 $48 $14 $50 $147 $33 $46 $48 $61 $10 $10 $21 $233 $31 $302 $57 $38 $156 $16 $48 $105 $1 $60 $1 $455 $6 $16 $182 $47 $128 $9 $25 $90 $128 $5 $101 $104 $19 $29 $75 $62 $72 $27 $47 $16 $30 $26 $101 $4 $112 $32 $157 $1 $77 $50 $66 $5 $42 $129 $284 $42 $5 $9 $74 $31 $15 $129 $207 $21 $186 $37 $71 $49 $27 $55 $26 $87 $11 $197 $39 $26 $36 $54 $28 $30 $24 $36 $54 $94 $2 $21 $58 $64 $16 $42 $49 $11 $9 $87 $39 $8 $160 $1 $102 $111 $64 $34 $323 $81 $2 $17 $95 $46 $81 $58 $86 $117 $39 $38 $8 $6 $5 $1 $29 $384 $46 $270 $31 $9 $36 $23 $7 $215 $29 $4 $64 $44 $4 $38 $6 $328 $22 $51 $88 $137 $63 $51 $73 $0 $41 $33 $61 $21 $10 $13 $24 $80 $40 $1 $17 

  0%|          | 0/200 [00:00<?, ?it/s]

In [22]:
# This is how to save the model if you want to, particularly if you run this on a larger dataset

# import joblib
# joblib.dump(rf_model, "random_forest.joblib")

## Introducing XGBoost

Like Random Forest, XGBoost is also an ensemble model that combines multiple decision trees.

But unlike Random Forest, XGBoost builds one tree after another, with each next tree correcting for errors in the prior trees, using 'gradient descent'.

It's much faster than Random Forest, so we can run it for the full dataset, and it's typically better at generalizing.

**If this import doesn't work, please skip this! It's not required. On a Mac, you might need to do `brew install libomp` in the terminal.**

In [23]:
import xgboost as xgb

In [24]:
np.random.seed(42)

xgb_model = xgb.XGBRegressor(n_estimators=1000, random_state=42, n_jobs=4, learning_rate=0.1)
xgb_model.fit(X, prices)

Out[1]: 
XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1000,
             n_jobs=4, num_parallel_tree=None, ...)


In [25]:
def xg_boost(item):
    x = vectorizer.transform([item.summary])
    return max(0, xgb_model.predict(x)[0])

In [26]:
evaluate(xg_boost, test)

$81 $71 $37 $36 $121 $152 $39 $21 $51 $18 $578 $238 $65 $115 $5 $4 $68 $20 $38 $14 $23 $11 $6 $51 $246 $319 $74 $35 $58 $28 $87 $64 $16 $10 $50 $18 $5 $61 $157 $65 $161 $74 $61 $27 $100 $93 $43 $58 $44 $3 $42 $48 $122 $19 $228 $80 $61 $188 $14 $37 $73 $0 $67 $32 $426 $47 $39 $281 $45 $93 $7 $39 $107 $119 $7 $189 $218 $4 $38 $91 $13 $94 $111 $40 $15 $51 $48 $130 $21 $246 $35 $128 $42 $6 $26 $110 $112 $27 $179 $198 $89 $37 $23 $66 $20 $93 $147 $214 $31 $84 $37 $58 $84 $55 $72 $117 $175 $33 $163 $40 $49 $105 $58 $49 $124 $7 $15 $93 $129 $52 $44 $64 $79 $35 $32 $71 $82 $59 $120 $70 $10 $141 $80 $115 $60 $67 $31 $302 $21 $6 $19 $147 $48 $61 $102 $99 $114 $32 $7 $25 $73 $21 $3 $13 $417 $2 $185 $50 $13 $8 $17 $16 $85 $26 $38 $45 $42 $36 $28 $70 $288 $21 $93 $32 $103 $52 $61 $55 $83 $32 $47 $27 $22 $18 $3 $33 $68 $56 $65 $24 

  0%|          | 0/200 [00:00<?, ?it/s]

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">Traditional ML isn't just useful for learning the history; it's still heavily used in industry today, particularly for tasks where there are clearly identifiable features. It's worth spending time exploring the algorithms and experimenting here. See if you can beat my numbers with Traditional ML! I ran the Random Forest for the entire 800,000 training dataset. It took about 15 hours to run, and it ended up getting a low error of $56.40. Traditional ML can do well - try it for yourself.</span>
        </td>
    </tr>
</table>